In [6]:
from google.colab import drive, files
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!ls -lh /content/drive/MyDrive | grep -i parquet


-rw------- 1 root root 178M Sep 22 19:52 df_merge_clean.parquet


In [10]:
import duckdb
con = duckdb.connect()
con.execute(f"""
CREATE OR REPLACE VIEW base AS
SELECT
  CAST(pdv AS BIGINT)      pdv,
  CAST(produto AS BIGINT)  produto,
  premise, categoria,
  iso_year, iso_week,
  CAST(quantity AS DOUBLE) qty
FROM read_parquet('{PARQUET}')
WHERE iso_year=2022
""")
con.execute("SELECT COUNT(*) AS rows, MIN(iso_week) w_min, MAX(iso_week) w_max FROM base").df()

,rows,w_min,w_max
0,6423199,1,52


In [9]:
import duckdb
con = duckdb.connect()
PARQUET = "/content/drive/MyDrive/df_merge_clean.parquet"

TARGET_SHARE = 0.98
OUT = "submission_s2c_int.parquet"

sql = f"""
WITH base AS (
  SELECT
    CAST(pdv AS BIGINT)      pdv,
    CAST(produto AS BIGINT)  produto,
    premise, categoria,
    iso_year, iso_week,
    CAST(quantity AS DOUBLE) qty
  FROM read_parquet('{PARQUET}')
  WHERE iso_year = 2022
),

-- volume recente por grupo e par
recent AS (
  SELECT premise, categoria, pdv, produto,
         SUM(qty) FILTER (WHERE iso_week BETWEEN 45 AND 52) AS qty_recent
  FROM base
  GROUP BY 1,2,3,4
),
ranked AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY premise, categoria ORDER BY qty_recent DESC) AS rn,
         SUM(qty_recent)  OVER (PARTITION BY premise, categoria)                        AS grp_sum,
         SUM(qty_recent)  OVER (PARTITION BY premise, categoria ORDER BY qty_recent DESC
                                 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)     AS cumsum_grp
  FROM recent
),
cut AS (
  SELECT premise, categoria, pdv, produto
  FROM ranked
  WHERE qty_recent > 0
    AND cumsum_grp / NULLIF(grp_sum,0) <= {TARGET_SHARE}
),

weekly AS (
  SELECT b.premise, b.categoria, b.pdv, b.produto, b.iso_week,
         SUM(qty) AS qty
  FROM base b
  JOIN cut  c USING (premise, categoria, pdv, produto)
  GROUP BY 1,2,3,4,5
),

-- seeds: weighted recent + fallbacks
wmean AS (
  SELECT premise, categoria, pdv, produto,
         -- pesos: 0.5, 0.3, 0.15, 0.05 nas semanas 52..49
         (0.50*AVG(qty) FILTER (WHERE iso_week=52)
        + 0.30*AVG(qty) FILTER (WHERE iso_week=51)
        + 0.15*AVG(qty) FILTER (WHERE iso_week=50)
        + 0.05*AVG(qty) FILTER (WHERE iso_week=49)) AS seed_wdecay,
         AVG(qty) FILTER (WHERE iso_week BETWEEN 49 AND 52)        AS seed_mean4,
         SUM(qty) FILTER (WHERE iso_week BETWEEN 45 AND 52)/8.0    AS seed_sum8
  FROM weekly
  GROUP BY 1,2,3,4
),
lnz8 AS (
  SELECT premise, categoria, pdv, produto,
         FIRST(qty) AS seed_lnz8
  FROM (
    SELECT w.*, ROW_NUMBER() OVER (PARTITION BY premise,categoria,pdv,produto
                                   ORDER BY iso_week DESC) AS rn
    FROM weekly w
    WHERE iso_week BETWEEN 45 AND 52 AND qty > 0
  ) t WHERE rn=1
  GROUP BY premise, categoria, pdv, produto
),

-- sazonalidade: Jan/2022 vs W45–52/2022 por grupo
season AS (
  SELECT premise, categoria,
         NULLIF(AVG(qty) FILTER (WHERE iso_week BETWEEN 1 AND 5),0)
         / NULLIF(AVG(qty) FILTER (WHERE iso_week BETWEEN 45 AND 52),0) AS s_jan_rel
  FROM base
  GROUP BY 1,2
),

seed0 AS (
  SELECT w.premise, w.categoria, w.pdv, w.produto,
         COALESCE(w.seed_wdecay, l.seed_lnz8, w.seed_mean4, w.seed_sum8, 0.1) AS qty_seed
  FROM wmean w
  LEFT JOIN lnz8 l USING (premise,categoria,pdv,produto)
),

-- clamp por p95 no grupo
caps AS (
  SELECT premise, categoria, quantile_cont(qty_seed, 0.95) AS p95
  FROM seed0
  GROUP BY 1,2
),

seed_adj AS (
  SELECT s.pdv, s.produto,
         GREATEST(0.0,
           LEAST(s.qty_seed * COALESCE(se.s_jan_rel,1.0),
                c.p95 * 1.05)
         ) AS quantidade
  FROM seed0 s
  LEFT JOIN season se USING (premise,categoria)
  LEFT JOIN caps   c  USING (premise,categoria)
),

weeks AS (SELECT * FROM (VALUES (1),(2),(3),(4),(5)) t(semana))

SELECT
  CAST(w.semana AS INT)       AS semana,
  CAST(sa.pdv AS BIGINT)      AS pdv,
  CAST(sa.produto AS BIGINT)  AS produto,
  CAST(ROUND(sa.quantidade) AS BIGINT) AS quantidade
FROM weeks w CROSS JOIN seed_adj sa
"""

sub = con.execute(sql).df()
print("rows:", len(sub), "| uniq keys:", sub[['semana','pdv','produto']].drop_duplicates().shape[0])

OUT_CMD = f"COPY ({sql}) TO '{OUT}' (FORMAT 'parquet')"
con.execute(OUT_CMD)
print("gravado:", OUT)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 1801755 | uniq keys: 1801755


RuntimeError: Query interrupted

In [20]:
import duckdb
try: con
except NameError: con = duckdb.connect()

TARGET_SHARE = 0.953
OUT = "submission_s2c_int_v5.parquet"

sql = f"""
WITH recent AS (
  SELECT premise, categoria, pdv, produto,
         SUM(qty) FILTER (WHERE iso_week BETWEEN 45 AND 52) AS qty_recent
  FROM base GROUP BY 1,2,3,4
),
ranked AS (
  SELECT *,
         SUM(qty_recent) OVER (PARTITION BY premise,categoria) AS grp_sum,
         SUM(qty_recent) OVER (PARTITION BY premise,categoria ORDER BY qty_recent DESC
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS csum
  FROM recent
),
cut AS (
  SELECT premise, categoria, pdv, produto
  FROM ranked
  WHERE qty_recent>0 AND csum/NULLIF(grp_sum,0) <= {TARGET_SHARE}
),
weekly AS (
  SELECT b.premise,b.categoria,b.pdv,b.produto,b.iso_week, SUM(qty) qty
  FROM base b JOIN cut c USING(premise,categoria,pdv,produto)
  GROUP BY 1,2,3,4,5
),
wmean AS (
  SELECT premise,categoria,pdv,produto,
         (0.50*AVG(qty) FILTER (WHERE iso_week=52)
        + 0.30*AVG(qty) FILTER (WHERE iso_week=51)
        + 0.15*AVG(qty) FILTER (WHERE iso_week=50)
        + 0.05*AVG(qty) FILTER (WHERE iso_week=49)) AS seed_wdecay,
         AVG(qty) FILTER (WHERE iso_week BETWEEN 49 AND 52)     AS seed_mean4,
         SUM(qty) FILTER (WHERE iso_week BETWEEN 45 AND 52)/8.0 AS seed_sum8
  FROM weekly GROUP BY 1,2,3,4
),
lnz8 AS (
  SELECT premise,categoria,pdv,produto, FIRST(qty) AS seed_lnz8
  FROM (
    SELECT w.*, ROW_NUMBER() OVER (PARTITION BY premise,categoria,pdv,produto
                                   ORDER BY iso_week DESC) rn
    FROM weekly w WHERE iso_week BETWEEN 45 AND 52 AND qty>0
  ) t WHERE rn=1
  GROUP BY premise,categoria,pdv,produto
),
season AS (
  SELECT premise,categoria,
         NULLIF(AVG(qty) FILTER (WHERE iso_week BETWEEN 1 AND 5),0)
         / NULLIF(AVG(qty) FILTER (WHERE iso_week BETWEEN 45 AND 52),0) AS s_jan_rel
  FROM base GROUP BY 1,2
),
seed0 AS (
  SELECT w.premise,w.categoria,w.pdv,w.produto,
         COALESCE(w.seed_wdecay, l.seed_lnz8, w.seed_mean4, w.seed_sum8, 0.1) AS qty_seed
  FROM wmean w LEFT JOIN lnz8 l USING(premise,categoria,pdv,produto)
),
caps AS (
  SELECT premise,categoria, quantile_cont(qty_seed,0.95) AS p95
  FROM seed0 GROUP BY 1,2
),
seed_adj AS (
  SELECT s.pdv, s.produto,
         GREATEST(0.0, LEAST(
           s.qty_seed * COALESCE(se.s_jan_rel,1.0),
           c.p95 * 1.05
         )) AS quantidade
  FROM seed0 s
  LEFT JOIN season se USING(premise,categoria)
  LEFT JOIN caps   c  USING(premise,categoria)
),
weeks AS (SELECT * FROM (VALUES (1),(2),(3),(4),(5)) t(semana))
SELECT
  CAST(semana AS INT)               AS "semana",
  CAST(pdv AS BIGINT)               AS "pdv",
  CAST(produto AS BIGINT)           AS "produto",
  CAST(ROUND(quantidade) AS BIGINT) AS "quantidade"
FROM weeks CROSS JOIN seed_adj
"""

# validação rápida
sub = con.execute(sql).df()
print("rows:", len(sub), "| dups:", sub.duplicated(['semana','pdv','produto']).sum())

# exporta direto do SQL
con.execute(f"COPY ({sql}) TO '{OUT}' (FORMAT 'parquet')")
print("gravado:", OUT)

# download para enviar na plataforma
from google.colab import files
files.download(OUT)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 1496780 | dups: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

gravado: submission_s2c_int_v4.parquet


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
# Corrigir colunas MAIÚSCULAS -> minúsculas e garantir inteiros

INP = "submission_s2c_int_v4.parquet"          # ajuste se o nome for outro
OUT = "submission_s2c_int_v4_lower.parquet"

import pandas as pd

df = pd.read_parquet(INP)
# renomeia exatamente como a plataforma exige
df = df.rename(columns={
    "Semana":"semana",
    "PDV":"pdv",
    "Produto":"produto",
    "Quantidade":"quantidade"
})

# garante tipos inteiros
for c in ["semana","pdv","produto","quantidade"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).round().astype("int64")

# sanity check rápido
print(df.dtypes)
print(df.head(3))

df.to_parquet(OUT, index=False)
print("gravado:", OUT)

# baixar no Colab
from google.colab import files
files.download(OUT)

semana        int64
pdv           int64
produto       int64
quantidade    int64
dtype: object
   semana                  pdv              produto  quantidade
0       1  2729023468201744038  6766242611562432056           3
1       1  1082822971566526371  8078263384137356777           8
2       1   437955528321342848  1300170522939725968          10
gravado: submission_s2c_int_v4_lower.parquet


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
# nomes e tipos
assert list(df.columns)==['semana','pdv','produto','quantidade']
assert str(df.dtypes['pdv'])=='int64' and str(df.dtypes['produto'])=='int64'
assert str(df.dtypes['semana'])=='int64' and str(df.dtypes['quantidade'])=='int64'

# sem NaN, sem negativos, sem duplicatas
assert df.isna().sum().sum()==0
assert (df[['semana','pdv','produto','quantidade']]>=0).all().all()
assert not df.duplicated(['semana','pdv','produto']).any()

In [25]:
import pyarrow.parquet as pq
print(pq.read_table('submission_s2c_int_v4_lower.parquet').schema)

semana: int64
pdv: int64
produto: int64
quantidade: int64
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 515
